In [43]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [44]:
file_path = "../data/Processed/furniture_returns_baseline.xlsx"

df = pd.read_excel(file_path)

print("Baseline dataset loaded successfully!")
print("Dataset shape:", df.shape)

Baseline dataset loaded successfully!
Dataset shape: (5057, 27)


In [45]:
required_columns = [
    "Return ID",
    "Product",
    "Value ₹",
    "Transit Days",
    "SLA hrs",
    "Baseline Score",
    "Baseline Priority",
    "Value Risk Score",
    "Severity Score",
    "Safety Score",
    "Transit Risk Score",
    "SLA Urgency Score",
    "Condition Risk Score"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:")
    print(missing_columns)
else:
    print("All required columns are available!")

All required columns are available!


In [46]:
priority_distribution = (
    df["Baseline Priority"]
    .value_counts()
    .to_frame("Count")
)

priority_distribution["Percentage"] = (
    priority_distribution["Count"]
    / len(df) * 100
).round(2)

priority_distribution

,Count,Percentage
Baseline Priority,,
Medium,2384,47.14
Low,2332,46.11
High,341,6.74


In [47]:
priority_analysis = (
    df.groupby("Baseline Priority")
    [
        [
            "Value ₹",
            "Transit Days",
            "SLA hrs",
            "Baseline Score"
        ]
    ]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

priority_analysis

Value ₹                                   Transit Days  \
                    count      mean   median    min     max        count   
Baseline Priority                                                          
High                  341  45190.06  43374.0  16885  526480          341   
Low                  2332  16104.97  12503.0      0   37406         2332   
Medium               2384  34305.09  33414.0      0  566460         2384   

                                        SLA hrs                        \
                   mean median min  max   count   mean median min max   
Baseline Priority                                                       
High               5.12    4.0   0   30     341  40.96   48.0  24  72   
Low                4.89    2.0   0  999    2332  51.03   48.0  24  72   
Medium             8.47    3.0   0  999    2384  45.12   48.0  24  72   

                  Baseline Score                          
                           count  mean median  min   max  
Baseline Priority                                         
High                         341  2.77   2.75  2.6  3.35  
Low                         2332  1.43   1.45  1.0  1.75  
Medium                      2384  2.10   2.10  1.8  2.55

In [48]:
value_tradeoff = (
    df.groupby("Baseline Priority")["Value ₹"]
    .agg(
        Count="count",
        Average_Value="mean",
        Median_Value="median",
        Total_Value="sum"
    )
    .round(2)
)

value_tradeoff

,Count,Average_Value,Median_Value,Total_Value
Baseline Priority,,,,
High,341,45190.06,43374.0,15409809
Low,2332,16104.97,12503.0,37556780
Medium,2384,34305.09,33414.0,81783346


In [49]:
sla_tradeoff = (
    df.groupby("Baseline Priority")["SLA hrs"]
    .agg(
        Count="count",
        Average_SLA="mean",
        Minimum_SLA="min",
        Maximum_SLA="max"
    )
    .round(2)
)

sla_tradeoff

,Count,Average_SLA,Minimum_SLA,Maximum_SLA
Baseline Priority,,,,
High,341,40.96,24,72
Low,2332,51.03,24,72
Medium,2384,45.12,24,72


In [50]:
transit_tradeoff = (
    df.groupby("Baseline Priority")["Transit Days"]
    .agg(
        Count="count",
        Average_Transit="mean",
        Median_Transit="median",
        Maximum_Transit="max"
    )
    .round(2)
)

transit_tradeoff

,Count,Average_Transit,Median_Transit,Maximum_Transit
Baseline Priority,,,,
High,341,5.12,4.0,30
Low,2332,4.89,2.0,999
Medium,2384,8.47,3.0,999


In [51]:
risk_tradeoff = (
    df.groupby("Baseline Priority")
    [
        [
            "Value Risk Score",
            "Severity Score",
            "Safety Score",
            "Transit Risk Score",
            "SLA Urgency Score",
            "Condition Risk Score"
        ]
    ]
    .mean()
    .round(2)
)

risk_tradeoff

,Value Risk Score,Severity Score,Safety Score,Transit Risk Score,SLA Urgency Score,Condition Risk Score
Baseline Priority,,,,,,
High,3.73,2.47,2.46,2.17,2.29,2.26
Low,1.66,1.14,1.12,1.54,1.87,1.24
Medium,3.03,1.58,1.54,1.85,2.12,1.63


In [52]:
HOLDING_COST_PER_DAY = 100

print(
    "Illustrative holding cost per return per day: ₹",
    HOLDING_COST_PER_DAY
)

Illustrative holding cost per return per day: ₹ 100


In [53]:
df["Estimated Holding Cost"] = (
    df["Transit Days"] *
    HOLDING_COST_PER_DAY
)

df[
    [
        "Return ID",
        "Transit Days",
        "Estimated Holding Cost"
    ]
].head(10)

,Return ID,Transit Days,Estimated Holding Cost
0,R2839,22,2200
1,R1477,13,1300
2,R0460,16,1600
3,R2519,7,700
4,R2638,13,1300
5,R1743,14,1400
6,R0121,6,600
7,R0980,12,1200
8,R1758,16,1600
9,R0973,6,600


In [54]:
holding_cost_analysis = (
    df.groupby("Baseline Priority")
    ["Estimated Holding Cost"]
    .agg(
        Count="count",
        Average_Cost="mean",
        Total_Cost="sum"
    )
    .round(2)
)

holding_cost_analysis

,Count,Average_Cost,Total_Cost
Baseline Priority,,,
High,341,512.02,174600
Low,2332,489.41,1141300
Medium,2384,847.36,2020100


In [55]:
df["Operational Exposure"] = (
    df["Value ₹"] +
    df["Estimated Holding Cost"]
)

df[
    [
        "Return ID",
        "Value ₹",
        "Estimated Holding Cost",
        "Operational Exposure"
    ]
].head(10)

,Return ID,Value ₹,Estimated Holding Cost,Operational Exposure
0,R2839,41143,2200,43343
1,R1477,44311,1300,45611
2,R0460,43938,1600,45538
3,R2519,43798,700,44498
4,R2638,38884,1300,40184
5,R1743,44482,1400,45882
6,R0121,37762,600,38362
7,R0980,54891,1200,56091
8,R1758,49846,1600,51446
9,R0973,42542,600,43142


In [56]:
highest_exposure = (
    df.sort_values(
        "Operational Exposure",
        ascending=False
    )
    [
        [
            "Return ID",
            "Product",
            "Value ₹",
            "Transit Days",
            "Estimated Holding Cost",
            "Operational Exposure",
            "Baseline Priority"
        ]
    ]
    .head(20)
)

highest_exposure

,Return ID,Product,Value ₹,Transit Days,Estimated Holding Cost,Operational Exposure,Baseline Priority
1061,R3225,bed,566460,6,600,567060,Medium
104,R2069,bed,526480,1,100,526580,High
1062,R3741,bed,470880,6,600,471480,Medium
26,R2288,bed,433280,0,0,433280,High
1160,R1935,bed,431990,4,400,432390,Medium
2131,R3481,sofa,407270,1,100,407370,Medium
59,R3230,wardrobe,200390,3,300,200690,High
786,R0041,coffee table,165330,0,0,165330,Medium
342,R1992,bed,58999,999,99900,158899,Medium
2132,R4381,coffee table,153520,1,100,153620,Medium


In [57]:
sla_urgency_distribution = (
    df["SLA Urgency Score"]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

sla_urgency_distribution["Percentage"] = (
    sla_urgency_distribution["Count"]
    / len(df) * 100
).round(2)

sla_urgency_distribution

,Count,Percentage
SLA Urgency Score,,
1,1544,30.53
2,1877,37.12
3,1636,32.35


In [58]:
safety_priority = pd.crosstab(
    df["Baseline Priority"],
    df["Safety"]
)

safety_priority

Safety,high,low,medium,unknown
Baseline Priority,,,,
High,198,37,103,3
Low,75,2017,141,99
Medium,414,1413,466,91


In [59]:
severity_priority = pd.crosstab(
    df["Baseline Priority"],
    df["Severity"]
)

severity_priority

Severity,high,low,medium,unknown
Baseline Priority,,,,
High,196,31,110,4
Low,69,804,194,1265
Medium,446,626,486,826


In [60]:
tradeoff_summary = pd.DataFrame({
    "Priority": ["High", "Medium", "Low"],
    "Inspection Focus": [
        "Immediate attention",
        "Planned inspection",
        "Standard queue"
    ],
    "Main Trade-off": [
        "Higher operational attention and faster handling",
        "Balance urgency and available resources",
        "Lower immediate resource allocation"
    ]
})

tradeoff_summary

,Priority,Inspection Focus,Main Trade-off
0,High,Immediate attention,Higher operational attention and faster handling
1,Medium,Planned inspection,Balance urgency and available resources
2,Low,Standard queue,Lower immediate resource allocation


In [61]:
print("=" * 60)
print("TRADE-OFF ANALYSIS OBSERVATIONS")
print("=" * 60)

print("\n1. High-priority returns receive greater inspection attention.")

print(
    "\n2. Product value, severity, safety, transit risk, "
    "SLA urgency and condition risk contribute to prioritization."
)

print(
    "\n3. Longer transit duration increases the estimated "
    "holding-cost exposure under the stated assumption."
)

print(
    "\n4. SLA urgency helps identify returns that may require "
    "faster operational handling."
)

print(
    "\n5. The prioritizer balances multiple operational signals "
    "instead of relying only on product value."
)

print(
    "\n6. Holding-cost estimates are illustrative assumptions "
    "and should be replaced with real business data when available."
)

TRADE-OFF ANALYSIS OBSERVATIONS

1. High-priority returns receive greater inspection attention.

2. Product value, severity, safety, transit risk, SLA urgency and condition risk contribute to prioritization.

3. Longer transit duration increases the estimated holding-cost exposure under the stated assumption.

4. SLA urgency helps identify returns that may require faster operational handling.

5. The prioritizer balances multiple operational signals instead of relying only on product value.

6. Holding-cost estimates are illustrative assumptions and should be replaced with real business data when available.


In [62]:
output_path = (
    "../data/Processed/"
    "tradeoff_analysis_results.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    priority_distribution.to_excel(
        writer,
        sheet_name="Priority Distribution"
    )

    value_tradeoff.to_excel(
        writer,
        sheet_name="Value Tradeoff"
    )

    sla_tradeoff.to_excel(
        writer,
        sheet_name="SLA Tradeoff"
    )

    transit_tradeoff.to_excel(
        writer,
        sheet_name="Transit Tradeoff"
    )

    risk_tradeoff.to_excel(
        writer,
        sheet_name="Risk Tradeoff"
    )

    holding_cost_analysis.to_excel(
        writer,
        sheet_name="Holding Cost"
    )

    highest_exposure.to_excel(
        writer,
        sheet_name="Highest Exposure",
        index=False
    )

print("Trade-off analysis results saved successfully!")
print(output_path)

Trade-off analysis results saved successfully!
../data/Processed/tradeoff_analysis_results.xlsx


In [63]:
print("=" * 60)
print("NOTEBOOK 07 COMPLETED")
print("=" * 60)

print("\nKey Areas Analysed:")
print("- Priority distribution")
print("- Product value")
print("- SLA urgency")
print("- Transit duration")
print("- Safety risk")
print("- Severity risk")
print("- Condition risk")
print("- Illustrative holding cost")
print("- Operational exposure")

print(
    "\nImportant Note:"
    "\nHolding cost is based on an illustrative assumption "
    "and is not presented as actual company cost."
)

NOTEBOOK 07 COMPLETED

Key Areas Analysed:
- Priority distribution
- Product value
- SLA urgency
- Transit duration
- Safety risk
- Severity risk
- Condition risk
- Illustrative holding cost
- Operational exposure

Important Note:
Holding cost is based on an illustrative assumption and is not presented as actual company cost.
